# MODULE 34 — SPEECH PROCESSING

# 34.8 Speech Recognition Fundamentals 🎙️

Bro, now we're connecting everything we've learned in speech processing to a real-world application: **Automatic Speech Recognition (ASR).**

Imagine you speak into your phone:

> "I want to learn machine learning."

The computer receives an audio signal, analyses its acoustic patterns, and predicts the words you spoke.

This process is called **Automatic Speech Recognition (ASR)**, commonly known as speech-to-text.

![Speech recognition research illustration](https://images.openai.com/static-rsc-4/HomEQKCsfgfjFevltRMDWW35Y3bGSBowygA6f91MEJk8VP940smP3KXrbCzYGHN79UMtEqH1asv47kGlkP1KgAzHNIcZ53MxkKJpcQyDJ1fI1DJteiKMIwUzu7e98bIvVNKgCxkJpARFHwMfvqCeX-0mNKYo0NN309XToawcpMM?purpose=inline)

Source: [Investigating the effects of gender, dialect, and training size on Arabic speech recognition — Springer Nature](https://link.springer.com/article/10.1007/s10579-020-09505-5)

---

## 1. What is Automatic Speech Recognition?

ASR is a technology that converts spoken language into written text.

- **Input:** Human speech (audio signal).
- **Output:** Recognized text (words or sentences).

**Examples:**

- Google Assistant converting speech into commands.
- YouTube generating captions.
- Meeting transcription software converting conversations into text.
- Voice typing on smartphones.
- Customer support systems transcribing calls.

An ASR system must solve several problems:

1. Identify the sounds present in speech.
2. Determine which language sounds and words are likely.
3. Understand the sequence of words.
4. Select the most likely sentence.

A major challenge is that the same word can sound different depending on the speaker, accent, speed, background noise and pronunciation.

---

## 2. ASR Pipeline

Let's understand the complete pipeline from your syllabus.

### Step-by-step process

**1. Human Speech**

A person speaks a sentence.

**2. Audio Recording**

Microphone captures the waveform.

**3. Audio Preprocessing**

Resampling, noise reduction and segmentation.

**4. Feature Extraction**

MFCC, Log-Mel spectrogram or learned features.

**5. Speech Recognition Model**

Acoustic representation → linguistic output.

**6. Text Prediction / Decoding**

Select the most likely word sequence.

**7. Recognized Sentence**

> "I want to learn machine learning."

One important distinction: this is a conceptual pipeline. Modern end-to-end systems can combine feature learning, acoustic modelling and text prediction inside one trained architecture rather than implementing every stage separately.

---

## 3. Traditional ASR Concepts

Before deep learning became dominant, speech recognition systems commonly used multiple statistical models working together.

The three major modelling components were:

1. Acoustic modelling.
2. Language modelling.
3. Pronunciation modelling.

Let's understand each.

### 3.1 Acoustic Modelling

An acoustic model learns the relationship between audio observations and speech units such as phonemes.

For example, when someone speaks:

> "CAT"

The audio contains acoustic patterns corresponding approximately to:

$$
/k/ \quad /æ/ \quad /t/
$$

The acoustic model estimates which phonemes or speech states are likely to have produced the observed audio.

- **Input:** Audio features, such as MFCCs.
- **Output:** Probabilities or scores for phonemes or acoustic states.

**Example:**

| Acoustic observation | Possible prediction |
|---|---|
| Sound pattern 1 | /k/ |
| Sound pattern 2 | /æ/ |
| Sound pattern 3 | /t/ |

The model doesn't necessarily identify complete words by itself. It contributes evidence about the sounds being spoken.

### 3.2 Language Modelling

An acoustic model might confuse words that sound similar.

For example:

- "I scream"
- "Ice cream"

The acoustic information may be similar, but the surrounding words help determine the intended sentence.

A language model learns the probability of word sequences.

$$
P(W)=P(w_1,w_2,\ldots,w_n)
$$

For example:

- "I want to eat ice cream." — likely sequence.
- "I want to eat I scream." — unlikely sequence in ordinary conversation.

The language model helps select linguistically plausible word sequences.

Its job: Use language patterns and context to improve transcription.

Traditional language models often used n-grams.

For a bigram model:

$$
P(w_1,w_2,\ldots,w_n)
\approx
\prod_{i=1}^{n}P(w_i|w_{i-1})
$$

This approximates each word's probability using the previous word.

### 3.3 Pronunciation Modelling

Pronunciation modelling describes how words are represented as sequences of phonemes.

For example:

| Word | Example phoneme representation |
|---|---|
| CAT | /k/ /æ/ /t/ |
| DOG | /d/ /ɒ/ /g/ |
| BOOK | /b/ /ʊ/ /k/ |

A pronunciation dictionary, also called a **lexicon**, maps words to their possible pronunciations.

**Why is this important?**

The acoustic model predicts speech sounds, but the recognizer needs to determine which words those sounds correspond to.

Pronunciation modelling connects the sound units to vocabulary words.

### 3.4 How the Three Components Work Together

![Attention and end-to-end ASR illustration](https://images.openai.com/static-rsc-4/LFS0ABviCcrRZA5--N1gq6u2I671A36hhfbXRSXeNz34rz1NPWO6DfoRBEclL-cJnaATBTu-uXEUakifBBeakAQALlOQVhdz21Rwkvp9shWfRE4d63dPMC6Q-2pNnSqzd-qmwKLD6EPLGuhBZ4gBKDPYLNoziCNdB2MPn9jIkxw?purpose=inline)

Source: [Attention in End-to-End Automatic Speech Recognition — Intel Student Ambassadors](https://medium.com/intel-student-ambassadors/attention-in-end-to-end-automatic-speech-recognition-9f9e42718d21)

| Component | Main question it answers |
|---|---|
| Acoustic model | Which sounds are present? |
| Pronunciation model | Which phoneme sequence represents this word? |
| Language model | Which word sequence makes sense? |

The decoder combines evidence from these components to find a likely transcription.

---

## 4. Basic Speech Units

Before exploring HMM and GMM, we need to understand four terms.

### 4.1 Phonemes

A phoneme is the smallest sound unit that can distinguish meaning in a language.

For example:

- CAT → /k/ /æ/ /t/
- BAT → /b/ /æ/ /t/

Changing `/k/` to `/b/` changes the word.

Notice that phonemes represent sounds, not necessarily individual written letters.

### 4.2 Graphemes

A grapheme is a written unit representing a sound or part of a word.

For example, in English:

- `c` in CAT.
- `b` in BAT.
- `sh` in SHIP.

A grapheme can consist of one or multiple letters.

| Term | Represents | Example |
|---|---|---|
| Phoneme | Spoken sound | /ʃ/ |
| Grapheme | Written representation | sh |

This distinction matters because speech recognition converts acoustic information into written language.

### 4.3 Vocabulary

Vocabulary refers to the collection of words that a recognition system can recognize or generate.

For example:

```text
Vocabulary:
- HELLO
- MACHINE
- LEARNING
- PYTHON
- ENGINEERING
```

Traditional systems often relied on a predefined vocabulary and pronunciation dictionary.

A word outside the vocabulary could be difficult for such a system to recognize correctly.

Modern systems can handle open-vocabulary situations more effectively using subword or character-based tokenization, although they still have limitations.

### 4.4 Word Decoding

The decoder determines the most likely word sequence based on the model's predictions.

A traditional ASR decoder combines acoustic, pronunciation and language-model evidence.

A simplified formulation is:

$$
\hat W=\arg\max_W P(W|X)
$$

Where:

- $X$ = Observed audio or acoustic features.
- $W$ = Candidate word sequence.
- $\hat W$ = Predicted word sequence.

Using Bayes' rule:

$$
\hat W=\arg\max_W P(X|W)P(W)
$$

Here:

- $P(X|W)$ represents acoustic evidence.
- $P(W)$ represents language-model probability.

This is a simplified formulation; practical decoders also account for pronunciation and search constraints.

---

## 5. Hidden Markov Models (HMM)

<badge color="success">Must Know — Traditional ASR</badge>

HMM stands for **Hidden Markov Model**.

It is a statistical model used to represent sequences where the actual underlying states are hidden, but observations generated by those states can be measured.

**Why is this useful in speech recognition?**

When someone speaks, we observe an audio signal. But we don't directly observe the phoneme states responsible for generating it.

HMM models these hidden speech states over time.

### 5.1 Understanding Hidden States

Consider someone speaking:

> CAT

The underlying phoneme sequence is approximately:

$$
/k/ \rightarrow /æ/ \rightarrow /t/
$$

We observe the audio waveform, but the phoneme states that generated it are hidden.

![Hidden Markov Model speech recognition illustration](https://images.openai.com/static-rsc-4/hzcOuXPx5xgf4lY8l-B87nvJJM-9ZqrQv4fbGoRdW_Apx2wjM8eg_AGITRmlt3Ff_enm1RUIkhkEAKDevFZjnoQhdwcQk1XWEoawkiOiQlawo9k9i307G0z99WX8YXf9Q68Doyn9xVOgmMFIiTB1AnEoZpbTvGSJwoKLDam0bTk?purpose=inline)

Source: [Introduction to Automatic Speech Recognition](https://maelfabien.github.io/machinelearning/speech_reco/)

In an HMM:

- **Hidden states:** Phonemes or phoneme sub-states.
- **Observations:** Acoustic feature vectors, such as MFCCs.
- **Transitions:** Probabilities of moving from one state to another.
- **Emissions:** Probabilities of observing an acoustic feature given a hidden state.

### 5.2 Two Important Probabilities

**1. Transition probability**

$$
P(q_t|q_{t-1})
$$

This represents the probability of moving from the previous hidden state to the current state.

Example:

$$
P(/æ/|/k/)
$$

It represents the probability of transitioning from the `/k/` state to the `/æ/` state.

**2. Emission probability**

$$
P(x_t|q_t)
$$

This represents the probability of observing an acoustic feature $x_t$ given the current hidden state $q_t$.

For example, how likely is a particular MFCC feature vector if the current phoneme state is `/k/`?

### 5.3 Markov Assumption

HMM uses the Markov assumption:

$$
P(q_t|q_1,\ldots,q_{t-1})
=
P(q_t|q_{t-1})
$$

This means the current hidden state depends only on the previous hidden state.

This is a simplifying assumption that makes sequential probability modelling computationally manageable.

### 5.4 Why is HMM Used in Traditional ASR?

Speech is sequential. One sound follows another, and sounds have different durations.

HMM provides a statistical framework for representing:

- Speech-state sequences.
- Transitions between speech states.
- Acoustic observations.
- Temporal alignment between audio frames and phonemes.

---

## 6. Gaussian Mixture Models (GMM)

GMM stands for **Gaussian Mixture Model**.

It is a probabilistic model that represents a data distribution as a weighted combination of multiple Gaussian distributions.

In traditional ASR, GMMs were commonly used to model the probability of acoustic feature vectors given an HMM state.

A Gaussian distribution is:

$$
\mathcal N(x|\mu,\Sigma)
$$

Where:

- $\mu$ = Mean vector.
- $\Sigma$ = Covariance matrix.
- $x$ = Observed feature vector.

A Gaussian mixture is:

$$
P(x)=
\sum_{m=1}^{M}
\pi_m\mathcal N(x|\mu_m,\Sigma_m)
$$

Where:

- $M$ = Number of Gaussian components.
- $\pi_m$ = Mixture weight.
- $\mu_m$ = Mean of component $m$.
- $\Sigma_m$ = Covariance of component $m$.

The mixture weights satisfy:

$$
\sum_{m=1}^{M}\pi_m=1
$$

**Intuition:** A single Gaussian may not represent the complex distribution of speech features adequately. Multiple Gaussian distributions can model different regions of that distribution.

### 6.1 How HMM and GMM Work Together

![GMM-HMM speech recognition illustration](https://images.openai.com/static-rsc-4/WwS_sivR3ndWNnJ1OJRySeOO_uk8f-tD3ODVx3KPRhY5K-UA2RjJ5yabOdL12noeObIc9D25Bi5UfjHJrhdUsO3fDCxAINjtSNYT-BNByavpcur4FOiLsPAZZWesaSn7pYD2zj79rlJJoTzJbHXAq6yxMnhgxNOWlsMefmzF1h4?purpose=inline)

Source: [IEIE Transactions on Smart Processing & Computing](https://ieiespc.org/ieiespc/XmlViewer/f450831)

In a traditional GMM-HMM speech recognition system:

| Component | Responsibility |
|---|---|
| HMM | Models transitions and hidden speech-state sequences |
| GMM | Models acoustic observation probabilities for each state |

This combination is called a **GMM-HMM acoustic model**.

It was a foundational architecture in traditional speech recognition.

---

## 7. Modern ASR Concepts

Traditional systems used separately designed statistical components. Modern ASR increasingly uses deep-learning architectures to learn speech representations and predict text.

Let's understand the main modern approaches.

### 7.1 Deep Learning-Based ASR

Deep-learning-based ASR uses neural networks to learn complex relationships between audio and text.

Instead of relying entirely on manually designed acoustic feature models such as GMMs, neural networks can learn useful acoustic representations from training data.

Common neural architectures include:

- CNNs.
- RNNs and LSTMs.
- Encoder-decoder networks.
- Transformers.

For example, a neural network can receive Log-Mel spectrograms and learn patterns associated with different speech sounds.

**Important:** Deep learning does not mean MFCCs or other handcrafted features are never used. Some systems still use them, while others learn representations directly from audio.

### 7.2 CTC — Connectionist Temporal Classification

<badge color="success">Must Know</badge>

CTC is a training objective and decoding approach designed for sequence tasks where the input and output sequences have different lengths and exact alignment is unknown.

This is particularly useful for speech recognition.

Consider:

**Input audio frames:**

```text
Frame 1 → Frame 2 → Frame 3 → Frame 4 → Frame 5
```

**Target text:**

```text
CAT
```

We don't know exactly which audio frame corresponds to each character. CTC handles this alignment problem.

CTC introduces a special blank token:

$$
\varnothing
$$

A model might predict:

```text
C → C → blank → A → T
```

CTC collapses repeated consecutive tokens and removes blank tokens:

```text
C C blank A T
       ↓
      CAT
```

Notice that the repeated `C` is collapsed before removing blanks.

For example:

```text
C blank C
   ↓
CC
```

The blank allows repeated characters to be represented correctly.

**Why is CTC useful?**

- Doesn't require manually aligned audio-text training examples.
- Handles different input and output sequence lengths.
- Supports efficient training of speech recognition models.

CTC is a way to train and decode sequence predictions; it is not itself a complete neural-network architecture.

### 7.3 Encoder-Decoder Architecture

Encoder-decoder architectures are widely used in sequence-to-sequence tasks.

The architecture has two main components.

![Encoder-decoder speech processing illustration](https://images.openai.com/static-rsc-4/MIfJxnyRx4hnPxsG6K2n0KTH4i6vnLMUsCI6ENQMpAlDh0SYDuTmAHuabg1Zk3aRowmygH3RDdb0Gj5gLK1fU08y29EPC9O_5xctzAVRXrDw2VnXAYAn6Z7SWsoo9KuFmTwU5sPXGW0YZN-zMNtOYnAnwxdbaKGZHqUSFsawzQI?purpose=inline)

Source: [Aman's AI Journal — Speech Processing](https://aman.ai/primers/ai/speech-processing/)

**Encoder:**

- Receives audio features.
- Extracts useful representations.
- Produces encoded information about the speech.

**Decoder:**

- Uses the encoded information.
- Predicts output tokens sequentially.
- Generates the recognized text.

Example:

```text
Audio:
[Audio features]

        ↓

      ENCODER

        ↓

Encoded representation

        ↓

      DECODER

        ↓

"I want to learn Python"
```

The encoder understands the acoustic input, while the decoder generates the corresponding text sequence.

### 7.4 Attention Mechanism

Attention helps a decoder focus on relevant parts of the encoded input while predicting each output token.

For example, when predicting the word "Python", the decoder can assign greater attention to the audio regions that contain evidence for that word.

![Attention mechanism illustration](https://images.openai.com/static-rsc-4/x6Dc0QT_LE4uQexP42KsZ9aIPUvB9qgfMAnWTW77Uclxc5Y1QAHiAaUJA4LKfmWb0G2u555TgEol5whi7RX8yVzVxyCSx-9MKiHF48x0PHQg3TUwMvXON8ZkqEmjl9BoFEel-bL_G4JwledmQ2a8XfI4RI6IUTCYqBq0WSZJph0?purpose=inline)

Source: [Models — Chiang Jun Wai](https://junwai7159.github.io/docs/note/natural_language_processing/speech/Models/)

The basic scaled dot-product attention formula is:

$$
Attention(Q,K,V)=
softmax\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

Where:

- $Q$ = Queries.
- $K$ = Keys.
- $V$ = Values.
- $d_k$ = Key dimension.

You've already studied attention in Transformers, so the important connection here is that attention can help connect different parts of the audio representation with the text being generated.

### 7.5 Transformer-Based ASR

Transformers use attention mechanisms to model relationships across sequence positions.

In speech recognition, Transformer architectures can process acoustic representations and generate text.

![Whisper Transformer ASR illustration](https://images.openai.com/static-rsc-4/iFdPwe5MBCj5mw7thzII8H4FfPAgt4c9RqP5LZXQHHYx6vmocGyHRXXGDQ7A2MHZGLMAIaIPEmHMiSfHYqYeW5QAGsSgUB08bqZZ1BttWCg8HptuF5NxOmpnmPxotqj3JDvhSwL6MFsu0JnFmzI9_12bWgmDe9xji1DPw3Ngi9w?purpose=inline)

Source: [Whisper: Functionality and Finetuning](https://medium.com/%40okezieowen/whisper-functionality-and-finetuning-ba7f9444f55a)

A typical encoder-decoder Transformer ASR system has:

**Audio encoder**

- Processes acoustic features.
- Uses self-attention to capture relationships across audio frames.
- Produces contextual representations.

**Text decoder**

- Predicts text tokens.
- Uses previous output tokens and encoder information.
- Generates the transcription.

Examples of Transformer-related ASR approaches include Whisper and encoder-decoder speech models.

Not every Transformer ASR system uses an encoder-decoder structure. Some use encoder-only architectures with CTC or other prediction heads.

### 7.6 End-to-End Speech Recognition

End-to-end ASR aims to learn the mapping from audio input directly to text output through a jointly trained neural system.

**Traditional ASR:**

```text
Audio
  ↓
Acoustic Model
  ↓
Pronunciation Lexicon
  ↓
Language Model
  ↓
Decoder
  ↓
Text
```

**End-to-end ASR:**

```text
Audio
  ↓
Neural Network
  ↓
Text Tokens
  ↓
Recognized Text
```

**Why is it called end-to-end?**

Because the system can be trained to optimize the audio-to-text prediction task jointly, instead of separately training and integrating every traditional component.

However, some end-to-end systems still use external language models, tokenization, or beam-search decoders.

### 7.7 Self-Supervised Speech Learning

This is an important modern concept.

Training ASR systems requires large amounts of audio, and manually transcribing all that audio is expensive.

Self-supervised learning uses large amounts of unlabelled audio to learn useful speech representations without requiring a human transcription for every recording.

A popular example is **wav2vec 2.0**.

![wav2vec 2.0 speech representation illustration](https://images.openai.com/static-rsc-4/iyVML33_AXCJ5orQ2mtW8L6aAXWQaECEa-713qEEyHXwKstihQl9V1tm5m0d2dUlppta8tAJE_ZrSoIJuSwRlwtiAjjQBbhlsgjWj4DY54vYc8NopyrQvdyLesbbqe8xL3UzlK7swLyoANC_h1vQ3qXMGEGjjc7q1f84UwnvEoo?purpose=inline)

Source: [Transforming Audio to Vector with wav2vec 2.0](https://demystifyml.co/transforming-audio-to-vector-with-wav2vec-2)

**Simplified workflow:**

1. Feed raw audio into a neural feature encoder.
2. Convert the audio into latent representations.
3. Mask portions of the latent sequence.
4. Train the model to identify the correct representations of masked regions using a contrastive learning objective.
5. Fine-tune the pretrained model with labelled speech transcripts for ASR.

**Why is this useful?**

- Learns speech representations from unlabelled audio.
- Reduces dependence on manually transcribed data during pretraining.
- Can improve performance when labelled speech data is limited.

Remember: self-supervised pretraining is not the same as completely unsupervised ASR. A model typically still needs an appropriate supervised fine-tuning or decoding strategy to produce accurate transcripts.

---

## 8. Traditional ASR vs Modern ASR

This is the most important comparison from this module.

![Traditional ASR architecture illustration](https://images.openai.com/static-rsc-4/U_eBFwVev7P6nRvGyig05CTPI_CQE2mwLsQlp3duNi9YLO981J1THDXoqTiWrG66fwfrWueLU8ZBmbT2elIY9FTwQu1JxmhjxVd_c5HCJGZ1_ldFaMFb4d8XtDsFehxc6SIdviXojizEOCDbE5G0k_v4GVzBz5iuaPuE44BSLZ0?purpose=inline)

![Whisper model illustration](https://images.openai.com/static-rsc-4/1rNyS7gbshqVHhfhNkGMP_d7J2NcoPWlGBCQx9iQDzWG_diivL7CcDfHmdc0jM9ywqJb9G8yQr0NFwiZbo3-KhA42LC-UMmD025PHMRXtOQfIp-yKAqSigvW6Cp3tZ3gaBuRwfv01m1yvu3xkRrqKe7GOKDNNAau12vqsUabdAI?purpose=inline)

![End-to-end ASR illustration](https://images.openai.com/static-rsc-4/a8G5wCFlM6GP1fQM0Y-oViiWAx6FX39dZWF0S0OPRQJvgF3zIjTCYot7xak4AkhpyS88OWv_OE_pDF0vc29QQp20UCSPZ6tNAGcrj13pT0z-cIS42f0af_vjb1XevLN91_2RGEPR1Vp_XkPQrDNzlOO0eT5bDzlXQVfKGM9N6tc?purpose=inline)

| Aspect | Traditional ASR | Modern ASR |
|---|---|---|
| Acoustic modelling | GMM-HMM | Neural networks |
| Speech features | MFCC, filterbank features | Log-Mel, learned features, raw audio in some models |
| Language modelling | Often separate statistical model | Integrated or external neural language model |
| Pronunciation | Explicit pronunciation lexicon | Can learn subword or character mappings |
| Alignment | HMM state alignment | CTC, attention, transducer approaches |
| Architecture | Multiple connected components | Often jointly trained neural architecture |
| Training | Statistical modelling and supervised training | Deep learning, supervised and self-supervised learning |
| Examples | GMM-HMM systems | Whisper, wav2vec 2.0-based ASR, Conformer-based systems |

One important clarification: Modern ASR is not a single architecture. CTC, attention-based encoder-decoder models and transducer models are different approaches. Transformers are one family of neural architectures used within modern ASR.

---

## 9. Real-World ASR Architecture Examples

### 9.1 Whisper

![Whisper model](https://images.openai.com/static-rsc-4/glY5Zp8UiVF1abLU9n67sen2hrnO2a8pUygywnoX--Yj6V2VZUrPgEdkLQu28BgfsuGnHopxEQTi_XPnx998vPoKqocnKx7y06qO8dYCmiFzy8plRkOLnpuPTju35YoDoNr7Qz3trqYqaVyg2doGaIs3JLJCzdHQVACfeMwCL_w?purpose=inline)

Transformer-based encoder-decoder model. Supports multilingual speech recognition, transcription and translation.

### 9.2 wav2vec 2.0

![wav2vec 2.0 model](https://images.openai.com/static-rsc-4/A1yEazoSotRtSRX9I75yo3zjyFHJsggEplnLOEKA6tBdh1uoDdv1QAM3G-OghxrD5_d3CMmVBY1lCKY7Y7XyyxGrNXbYAuLe9RIoyEFLKb_FTODTcOfRcNQiLy2CshZd62C_YEl4GHttCV2w5CJWyXKgFvc0KM3QHFpDv-MsSBI?purpose=inline)

Self-supervised speech representation learning, followed by fine-tuning for speech recognition.

### 9.3 Conformer

![Conformer speech model](https://images.openai.com/static-rsc-4/QkhgWWYSRGiiXUYo1kekhugrjuVGwXiq84u7-WrgzjhIlcfRzaa2LCZwMqv93XY2p1cPXwCDltugFpJJLy8hbTCwFBoX2L42iAHFiKc7sXRWLytTDy72y9XcDLjs7bxhteTv19KvBboD0iSCM2thlK16pLhwb4XbVelT4UMJrxs?purpose=inline)

Combines convolution and self-attention to capture local and global patterns in speech.

---

## 10. What You Should Remember 🧠

1. ASR converts spoken language into written text.
2. Acoustic modelling identifies likely speech sounds.
3. Language modelling estimates likely word sequences.
4. Pronunciation modelling maps words to phonemes.
5. Phonemes represent sounds; graphemes represent written units.
6. HMM models hidden speech-state sequences and their transitions.
7. GMM models acoustic feature distributions.
8. CTC handles unknown alignment between input audio frames and output tokens.
9. Encoder-decoder architectures map audio representations to text sequences.
10. Attention helps models focus on relevant input information.
11. Transformers are widely used in modern ASR.
12. Self-supervised learning allows models to learn from unlabelled audio.

